# 歌词分词，词性标注

In [1]:
import json
import pandas as pd


# import thulac


from collections import Counter
from openai import OpenAI

In [20]:
# 可以选择是否加载
# jieba.load_userdict('data/mayday_dict_simple.txt')

In [60]:
import sys, os
sys.path.append('..')

# 分词，词频与词性分析

In [3]:
word_to_fix = {
    '阮': 'r',
    '袂': 'v',
    '春娇': 'n',
    '学会': 'v'
}

In [4]:
# def process_lyrics_with_jieba(text):
#     # 1. 词性标注与分词
#     # jieba.posseg 会同时返回词和词性
#     words_with_pos = pseg.cut(text)

    
#     # 2. 过滤无意义字符（标点、空格、单字符停用词）
#     filtered_data = []
#     for word, pos in words_with_pos:
#         # 排除标点符号（x表示标点）及空白字符
#         if pos != 'x' and len(word.strip()) > 0:
#             if word in word_to_fix:
#                 filtered_data.append((word, word_to_fix[word]))
#             else:
#                 filtered_data.append((word, pos))
    
#     # 3. 统计词频
#     word_counts = Counter([item[0] for item in filtered_data])
    
#     # 4. 汇总信息 (词, 词性, 频数)
#     # 我们以词为 Key，存储词性
#     word_pos_map = {word: pos for word, pos in filtered_data}
    
#     # 排序：按词频从高到低
#     sorted_results = []
#     for word, count in word_counts.most_common():
#         sorted_results.append({
#             "word": word,
#             "pos": word_pos_map[word],
#             "freq": count # 词频
#         })
    
#     return sorted_results

In [5]:
import re
from hanlp_restful import HanLPClient


api = "https://hanlp.com/hanlp/v21/redirect"
# api = "https://hanlp.hankcs.com/api"
# api = "https://www.hanlp.com/api"
HanLP = HanLPClient(api, auth="699691e7eaf61a3aca90d7b8", language='zh')


def is_chinese_word(word):
    """
    判断是否为纯中文词
    """
    return 1 if re.fullmatch(r'[\u4e00-\u9fff]+', word) else 0

def is_english_word(word):
    """
    判断是否为纯英文词
    """
    return 1 if re.fullmatch(r'[a-zA-Z]+', word) else 0



def process_lyrics_with_hanlp_multi_pos(text, word_to_fix=None):
    if not text:
        return []

    # 调用 HanLP
    result = HanLP.parse(text, tasks='pos/pku')

    sentences = result['tok/fine']
    pos_sentences = result['pos/pku']

    # 统计 (word, pos) -> freq
    word_pos_counter = Counter()

    for words, pos_tags in zip(sentences, pos_sentences):
        for word, tag in zip(words, pos_tags):

            word = word.strip()

            # 过滤标点
            if tag == 'w' or not word:
                continue

            # 词性修正
            if word_to_fix and word in word_to_fix:
                tag = word_to_fix[word]

            word_pos_counter[(word, tag)] += 1

    # 构建结果列表
    results = []
    for (word, pos), freq in word_pos_counter.items():
        results.append({
            "word": word,
            "pos": pos,
            "freq": freq,
            "is_chinese": is_chinese_word(word)
        })

    # 按词频排序
    results.sort(key=lambda x: x["freq"], reverse=True)

    return results


In [6]:
# thu = thulac.thulac(seg_only=False, filt=True) 

# def process_lyrics_with_thulac(text, word_to_fix=None):
#     if not text:
#         return []
    
#     # 2. 执行分词与词性标注
#     # 返回格式为 [[word, pos], [word, pos], ...]
#     words_with_pos = thu.cut(text)
    
#     # 3. 过滤无意义字符与词性修正
#     # thulac 的标点词性通常是 'w'
#     filtered_data = []
#     for word, pos in words_with_pos:
#         word = word.strip()
#         # 排除标点符号、空白字符
#         if pos != 'w' and len(word) > 0:
#             # 逻辑修正：word_to_fix 通常是修正词性
#             if word_to_fix and word in word_to_fix:
#                 filtered_data.append((word, word_to_fix[word]))
#             else:
#                 filtered_data.append((word, pos))
    
#     # 4. 统计词频
#     word_counts = Counter([item[0] for item in filtered_data])
    
#     # 5. 汇总信息
#     # 建立 word -> pos 映射
#     word_pos_map = {word: pos for word, pos in filtered_data}
    
#     sorted_results = []
#     for word, count in word_counts.most_common():
#         sorted_results.append({
#             "word": word,
#             "pos": word_pos_map[word],
#             "freq": count
#         })
    
#     return sorted_results

In [64]:
def lyric_words_process(path_prefix, word_to_fix=None):
    lyric_file_path = path_prefix + 'cleared_lyric_data.json'

    # 已解析的内容
    songs_id_exist = []
    if os.path.exists(path_prefix + "raw_words_data.csv"):
        df_exist = pd.read_csv(path_prefix + "raw_words_data.csv")
        songs_id_exist = df_exist['song_id'].unique().tolist()

    # 读取歌词文件
    with open(lyric_file_path, 'r') as f:
        lyric_data = json.load(f)
    lyric_words_dict = {}
    for i in lyric_data:
        if i and i['song_id'] not in songs_id_exist:
            # lyric_words_dict[i['song_id']] = process_lyrics_with_jieba(
            #     i['lyrics_text'])
            # lyric_words_dict[i['song_id']] = process_lyrics_with_thulac(
            #     i['lyrics_text'], word_to_fix=word_to_fix)
            print(i['song_name'])
            lyric_words_dict[i['song_id']] = process_lyrics_with_hanlp_multi_pos(
                i['lyrics_text'], word_to_fix=word_to_fix)
    rows = []
    for song_id, word_list in lyric_words_dict.items():
        for item in word_list:
            # 创建新字典，保留原始数据并加入歌曲ID列
            new_row = {
                'song_id': song_id,
                'word': item['word'],
                'pos': item['pos'],
                'freq': item['freq']
            }
            rows.append(new_row)

    # 3. 转换为 DataFrame
    df_word = pd.DataFrame(rows)
    # 合并df_exis和 df_word
    if 'df_exist' in locals():
        df_word = pd.concat([df_exist, df_word], ignore_index=True)
    return df_word

In [8]:
def words_data_merge(df_word, df_songs):
    # 合并
    # 1. 确保 df_word 的 song_id 是字符串
    df_word = df_word.copy()
    df_songs = df_songs.copy()
    df_word['song_id'] = df_word['song_id'].astype(str)
    df_word['word'] = df_word['word'].astype(str)
    df_word['is_chinese'] = df_word['word'].apply(is_chinese_word)
    df_word['is_english'] = df_word['word'].apply(is_english_word)
    # 把英文词转为小写
    df_word.loc[df_word['is_english'] == 1, 'word'] = df_word.loc[df_word['is_english'] == 1, 'word'].str.lower()
    # 2. 确保 df_unique 的 song_id 是字符串（并去掉可能存在的空格）
    df_songs['song_id'] = df_songs['song_id'].astype(str).str.strip()

    # 3. 执行合并
    df_merged = df_word.merge(df_songs, on='song_id', how='left')

    # 4. 删除空值
    # df_merged = df_merged.dropna()

    return df_merged

# 批量采集

In [ ]:
singers = [('luodayou', '罗大佑'), ('lizongsheng', '李宗盛'), ('zhangxueyou', '张学友'), ('twins', 'Twins'), ('wangsulong', '汪苏泷'), ('panweibo', '潘玮柏'), ('dengziqi', 'G.E.M. 邓紫棋'), ('xuezhiqian', '薛之谦'), ('xusong', '许嵩'), ('zhangjie', '张杰'), ('taozhe', '陶喆'), ('fangdatong', '方大同'), ('wangfei', '王菲'), ('maobuyi', '毛不易'), ('beyond', 'BEYOND')]
for i in singers[-1:]:
    file_path_prefix = f'data/{i[0]}/'
    # 歌曲数据
    df_songs = pd.read_csv(file_path_prefix + "cleared_song_data.csv")
    # 词性解析
    # hanlp暂时不需要word_to_fix
    df_word = lyric_words_process(file_path_prefix, word_to_fix=None)
    df_word.to_csv(file_path_prefix + "raw_words_data.csv", index=False)
    # 重新读取
    df_word_read = pd.read_csv(file_path_prefix + "raw_words_data.csv")
    df_merged = words_data_merge(df_word_read, df_songs)
    df_merged = df_merged.dropna(subset='song_name')
    # 过滤中文词
    df_merged_chn = df_merged[df_merged['is_chinese'] == 1]
    # 虚拟专辑数据
    df_songs_part = df_merged_chn[[
        'song_name_pure'
    ]].drop_duplicates(keep='first').reset_index(drop=True)
    # 只保留120个
    df_songs_part = df_songs_part.head(100)
    df_songs_part['album_name'] = "PART " + (df_songs_part.index // 10 +
                                            1).astype(str)
    df_songs_part['album_order'] = df_songs_part.index // 10
    # 虚拟专辑数据，index//12+1作为虚拟专辑
    df_merged_chn = df_merged_chn.copy()
    df_merged_chn['album_name_raw'] = df_merged_chn['album_name']
    df_merged_chn = df_merged_chn.drop(columns=['album_name'])
    df_merged_chn = df_merged_chn.merge(df_songs_part, on='song_name_pure', how='left')
    # 删除album_order为空的数据
    df_merged_chn = df_merged_chn.dropna(subset=['album_order'], axis=0)
    df_merged_chn.to_csv(file_path_prefix + "cleared_words_data.csv", index=False)
    df_songs_final = df_merged_chn.drop(columns=['word', 'pos', 'freq', 'is_chinese']).drop_duplicates().reset_index(drop=True)
    df_songs_final.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)

# main

In [53]:
singer_list = [
        'mayday', 'jaychou', 'liyuchun', 'chenyixun', 'renxianqi', 'linjunjie',
        'sunyanzi', 'remen', 'fangwenshan', 'chenxinhong', 'caiyilin', 'wubai', 'zhoushen', 'zhoushen_pure', 'fenghuangchuanqi', 'wanglihong', 'liangjingru', 'wangxinling', 'twins', 'beyond', 'wuyuetian', 'luodayou', 'fangdatong', 'taozhe', 'lizongsheng', 'mowenwei', 'fangdatong', 'wangsulong', 'maobuyi', 'zhoujielun', 'suyoupeng'
    ]
file_path_prefix = f"data/{singer_list[-1]}/"
file_path_prefix = f"data/maobuyi/"

In [56]:
# 歌曲数据
df_songs = pd.read_csv(file_path_prefix + "cleared_song_data.csv")
df_songs

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,song_name_pure,album_name_pure,publish_date,publish_year
0,251875009,001Cnurq0Oe2cM,一程山路,NaN,毛不易,1507534,001BHDR33FZVZ0,小王,9495273,001OFJ154OfZuW,215,1582128000,一程山路,一程山路,小王,2020-02-20,2020
1,336582682,002QhULf16tWw1,无名的人,《雄狮少年》电影主题曲,毛不易,1507534,001BHDR33FZVZ0,无名的人,24100646,002mPgSG01LLtu,282,1639411200,无名的人,无名的人,无名的人,2021-12-14,2021
2,203451421,003kLvu04bLGzi,消愁 (Live),NaN,毛不易,1507534,001BHDR33FZVZ0,明日之子 第7期,2188465,002xoonH2Bk7FR,179,1501257600,消愁,消愁,明日之子 第7期,2017-07-29,2017
3,203514624,00375L600p9sxv,像我这样的人 (Live),NaN,毛不易,1507534,001BHDR33FZVZ0,明日之子 第8期,2196371,0001n7a82gh6IY,171,1501862400,像我这样的人,像我这样的人,明日之子 第8期,2017-08-05,2017
4,254554296,002XkEH930NXSr,一荤一素 (Live),NaN,毛不易,1507534,001BHDR33FZVZ0,歌手·当打之年 第2期,10635028,002VxplL2gXAuH,314,1581609600,一荤一素,一荤一素,歌手·当打之年 第2期,2020-02-14,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124,503421737,0033mKxH3ELFKK,有你，就有新回忆,假日酒店Holiday Inn品牌主题曲,毛不易,1507534,001BHDR33FZVZ0,有你，就有新回忆 (假日酒店Holiday Inn品牌主题曲),53458324,000LvMNT3h6yVa,222,1713974400,有你就有新回忆,有你就有新回忆,有你，就有新回忆 (假日酒店Holiday Inn品牌主题曲),2024-04-25,2024
125,503409738,003VEsBQ1E8cdm,世间美好与你环环相扣 (2022壬寅年中央广播电视总台元宵晚会现场),NaN,毛不易,0,0032fmHO2UDnV3,NaN,0,NaN,141,1644854400,世间美好与你环环相扣,世间美好与你环环相扣,NaN,2022-02-15,2022
126,345070442,001tbY85093M7Q,易燃易爆炸,NaN,毛不易,1507534,001BHDR33FZVZ0,NaN,0,NaN,201,1483200000,易燃易爆炸,易燃易爆炸,NaN,2017-01-01,2017
127,292111304,0004EcHX2CB6GT,得过且过的勇者 (Live),NaN,毛不易,1507534,001BHDR33FZVZ0,2020最美的夜bilibili晚会,16601204,0002VXnS2aXARC,196,1609344000,得过且过的勇者,得过且过的勇者,2020最美的夜bilibili晚会,2020-12-31,2020


In [ ]:
# 五月天需要使用word_to_fix
# if file_path_prefix == "data/mayday/":
#     df_word = lyric_words_process(file_path_prefix, word_to_fix)
# else:
#     df_word = lyric_words_process(file_path_prefix, word_to_fix=None)

In [65]:
# 词性解析
# hanlp暂时不需要word_to_fix
df_word = lyric_words_process(file_path_prefix, word_to_fix=None)
df_word.to_csv(file_path_prefix + "raw_words_data.csv", index=False)
df_word

,song_id,word,pos,freq
0,251875009,的,u,7
1,251875009,着,u,6
2,251875009,不,d,6
3,251875009,你,r,6
4,251875009,了,u,5
...,...,...,...,...
11740,212809455,就是,v,1
11741,212809455,这样,r,1
11742,212809455,怎么,r,1
11743,212809455,一不小心,l,1


In [67]:
# 重新读取
df_word_read = pd.read_csv(file_path_prefix + "raw_words_data.csv")
df_word_read

,song_id,word,pos,freq
0,251875009,的,u,7
1,251875009,着,u,6
2,251875009,不,d,6
3,251875009,你,r,6
4,251875009,了,u,5
...,...,...,...,...
11740,212809455,就是,v,1
11741,212809455,这样,r,1
11742,212809455,怎么,r,1
11743,212809455,一不小心,l,1


In [68]:
df_merged = words_data_merge(df_word_read, df_songs)
df_merged = df_merged.dropna(subset='song_name')
df_merged

,song_id,word,pos,freq,is_chinese,is_english,song_mid,song_name,song_subname,artist_name,...,album_name,album_id,album_mid,duration,publish_time,song_name_unique,song_name_pure,album_name_pure,publish_date,publish_year
0,251875009,的,u,7,1,0,001Cnurq0Oe2cM,一程山路,NaN,毛不易,...,小王,9495273,001OFJ154OfZuW,215,1582128000,一程山路,一程山路,小王,2020-02-20,2020
1,251875009,着,u,6,1,0,001Cnurq0Oe2cM,一程山路,NaN,毛不易,...,小王,9495273,001OFJ154OfZuW,215,1582128000,一程山路,一程山路,小王,2020-02-20,2020
2,251875009,不,d,6,1,0,001Cnurq0Oe2cM,一程山路,NaN,毛不易,...,小王,9495273,001OFJ154OfZuW,215,1582128000,一程山路,一程山路,小王,2020-02-20,2020
3,251875009,你,r,6,1,0,001Cnurq0Oe2cM,一程山路,NaN,毛不易,...,小王,9495273,001OFJ154OfZuW,215,1582128000,一程山路,一程山路,小王,2020-02-20,2020
4,251875009,了,u,5,1,0,001Cnurq0Oe2cM,一程山路,NaN,毛不易,...,小王,9495273,001OFJ154OfZuW,215,1582128000,一程山路,一程山路,小王,2020-02-20,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11740,212809455,就是,v,1,1,0,000Zn1w34VHamY,说散就散,NaN,毛不易,...,NaN,0,NaN,89,857145600,说散就散,说散就散,NaN,1997-03-01,1997
11741,212809455,这样,r,1,1,0,000Zn1w34VHamY,说散就散,NaN,毛不易,...,NaN,0,NaN,89,857145600,说散就散,说散就散,NaN,1997-03-01,1997
11742,212809455,怎么,r,1,1,0,000Zn1w34VHamY,说散就散,NaN,毛不易,...,NaN,0,NaN,89,857145600,说散就散,说散就散,NaN,1997-03-01,1997
11743,212809455,一不小心,l,1,1,0,000Zn1w34VHamY,说散就散,NaN,毛不易,...,NaN,0,NaN,89,857145600,说散就散,说散就散,NaN,1997-03-01,1997


In [69]:
df_merged[df_merged['pos'] == 'e']['word'].unique()

array(['啊', '嘿', '唔', '唉', '噢', '哦', '呵呵，', '呵呵', '啊哈哈', '呼'],
      dtype=object)

In [70]:
# 过滤中文词
df_merged_chn = df_merged[df_merged['is_chinese'] == 1]
df_merged_chn
# 不过滤中文词
# df_merged_chn = df_merged.copy()
# 只保留中文和英文
# df_merged_chn = df_merged_chn[(df_merged_chn['is_chinese'] == 1) | (df_merged_chn['is_english'] == 1)]

,song_id,word,pos,freq,is_chinese,is_english,song_mid,song_name,song_subname,artist_name,...,album_name,album_id,album_mid,duration,publish_time,song_name_unique,song_name_pure,album_name_pure,publish_date,publish_year
0,251875009,的,u,7,1,0,001Cnurq0Oe2cM,一程山路,NaN,毛不易,...,小王,9495273,001OFJ154OfZuW,215,1582128000,一程山路,一程山路,小王,2020-02-20,2020
1,251875009,着,u,6,1,0,001Cnurq0Oe2cM,一程山路,NaN,毛不易,...,小王,9495273,001OFJ154OfZuW,215,1582128000,一程山路,一程山路,小王,2020-02-20,2020
2,251875009,不,d,6,1,0,001Cnurq0Oe2cM,一程山路,NaN,毛不易,...,小王,9495273,001OFJ154OfZuW,215,1582128000,一程山路,一程山路,小王,2020-02-20,2020
3,251875009,你,r,6,1,0,001Cnurq0Oe2cM,一程山路,NaN,毛不易,...,小王,9495273,001OFJ154OfZuW,215,1582128000,一程山路,一程山路,小王,2020-02-20,2020
4,251875009,了,u,5,1,0,001Cnurq0Oe2cM,一程山路,NaN,毛不易,...,小王,9495273,001OFJ154OfZuW,215,1582128000,一程山路,一程山路,小王,2020-02-20,2020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11740,212809455,就是,v,1,1,0,000Zn1w34VHamY,说散就散,NaN,毛不易,...,NaN,0,NaN,89,857145600,说散就散,说散就散,NaN,1997-03-01,1997
11741,212809455,这样,r,1,1,0,000Zn1w34VHamY,说散就散,NaN,毛不易,...,NaN,0,NaN,89,857145600,说散就散,说散就散,NaN,1997-03-01,1997
11742,212809455,怎么,r,1,1,0,000Zn1w34VHamY,说散就散,NaN,毛不易,...,NaN,0,NaN,89,857145600,说散就散,说散就散,NaN,1997-03-01,1997
11743,212809455,一不小心,l,1,1,0,000Zn1w34VHamY,说散就散,NaN,毛不易,...,NaN,0,NaN,89,857145600,说散就散,说散就散,NaN,1997-03-01,1997


In [71]:
# 查看歌曲数
df_merged_chn['song_name_pure'].nunique()

125

In [72]:
# 虚拟专辑数据
df_songs_part = df_merged_chn[[
    'song_name_pure'
]].drop_duplicates(keep='first').reset_index(drop=True)
# 只保留120个
df_songs_part = df_songs_part.head(100)
df_songs_part['album_name'] = "PART " + (df_songs_part.index // 10 +
                                         1).astype(str)
df_songs_part['album_order'] = df_songs_part.index // 10
df_songs_part

,song_name_pure,album_name,album_order
0,一程山路,PART 1,0
1,无名的人,PART 1,0
2,消愁,PART 1,0
3,像我这样的人,PART 1,0
4,一荤一素,PART 1,0
...,...,...,...
95,平凡的雕琢,PART 10,9
96,项羽虞姬,PART 10,9
97,小小的我,PART 10,9
98,看我始终,PART 10,9


In [73]:
# 虚拟专辑数据，index//12+1作为虚拟专辑
df_merged_chn = df_merged_chn.copy()
df_merged_chn['album_name_raw'] = df_merged_chn['album_name']
df_merged_chn = df_merged_chn.drop(columns=['album_name'])
df_merged_chn = df_merged_chn.merge(df_songs_part, on='song_name_pure', how='left')
# 删除album_order为空的数据
df_merged_chn = df_merged_chn.dropna(subset=['album_order'], axis=0)
df_merged_chn

,song_id,word,pos,freq,is_chinese,is_english,song_mid,song_name,song_subname,artist_name,...,duration,publish_time,song_name_unique,song_name_pure,album_name_pure,publish_date,publish_year,album_name_raw,album_name,album_order
0,251875009,的,u,7,1,0,001Cnurq0Oe2cM,一程山路,NaN,毛不易,...,215,1582128000,一程山路,一程山路,小王,2020-02-20,2020,小王,PART 1,0.0
1,251875009,着,u,6,1,0,001Cnurq0Oe2cM,一程山路,NaN,毛不易,...,215,1582128000,一程山路,一程山路,小王,2020-02-20,2020,小王,PART 1,0.0
2,251875009,不,d,6,1,0,001Cnurq0Oe2cM,一程山路,NaN,毛不易,...,215,1582128000,一程山路,一程山路,小王,2020-02-20,2020,小王,PART 1,0.0
3,251875009,你,r,6,1,0,001Cnurq0Oe2cM,一程山路,NaN,毛不易,...,215,1582128000,一程山路,一程山路,小王,2020-02-20,2020,小王,PART 1,0.0
4,251875009,了,u,5,1,0,001Cnurq0Oe2cM,一程山路,NaN,毛不易,...,215,1582128000,一程山路,一程山路,小王,2020-02-20,2020,小王,PART 1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9499,513744331,在,p,1,1,0,000sOitQ02o5rk,晚安曲 (Zzz),NaN,毛不易,...,212,1727056800,晚安曲,晚安曲,冒险精神 (The Inner Venture),2024-09-23,2024,冒险精神 (The Inner Venture),PART 10,9.0
9500,513744331,身上,s,1,1,0,000sOitQ02o5rk,晚安曲 (Zzz),NaN,毛不易,...,212,1727056800,晚安曲,晚安曲,冒险精神 (The Inner Venture),2024-09-23,2024,冒险精神 (The Inner Venture),PART 10,9.0
9501,513744331,入,v,1,1,0,000sOitQ02o5rk,晚安曲 (Zzz),NaN,毛不易,...,212,1727056800,晚安曲,晚安曲,冒险精神 (The Inner Venture),2024-09-23,2024,冒险精神 (The Inner Venture),PART 10,9.0
9502,513744331,梦乡,n,1,1,0,000sOitQ02o5rk,晚安曲 (Zzz),NaN,毛不易,...,212,1727056800,晚安曲,晚安曲,冒险精神 (The Inner Venture),2024-09-23,2024,冒险精神 (The Inner Venture),PART 10,9.0


In [74]:
df_merged_chn.to_csv(file_path_prefix + "cleared_words_data.csv", index=False)

In [75]:
# 数据查验
songs_n = df_merged_chn[df_merged_chn['pos'] == 'n']['song_name_pure'].unique().tolist()
songs_all = df_merged_chn['song_name_pure'].unique().tolist()
for i in songs_all:
    if i not in songs_n:
        print(i)

# 歌曲数据更新

In [76]:
df_songs_final = df_merged_chn.drop(columns=['word', 'pos', 'freq', 'is_chinese']).drop_duplicates().reset_index(drop=True)

df_songs_final

,song_id,is_english,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_id,album_mid,duration,publish_time,song_name_unique,song_name_pure,album_name_pure,publish_date,publish_year,album_name_raw,album_name,album_order
0,251875009,0,001Cnurq0Oe2cM,一程山路,NaN,毛不易,1507534,001BHDR33FZVZ0,9495273,001OFJ154OfZuW,215,1582128000,一程山路,一程山路,小王,2020-02-20,2020,小王,PART 1,0.0
1,336582682,0,002QhULf16tWw1,无名的人,《雄狮少年》电影主题曲,毛不易,1507534,001BHDR33FZVZ0,24100646,002mPgSG01LLtu,282,1639411200,无名的人,无名的人,无名的人,2021-12-14,2021,无名的人,PART 1,0.0
2,203451421,0,003kLvu04bLGzi,消愁 (Live),NaN,毛不易,1507534,001BHDR33FZVZ0,2188465,002xoonH2Bk7FR,179,1501257600,消愁,消愁,明日之子 第7期,2017-07-29,2017,明日之子 第7期,PART 1,0.0
3,203514624,0,00375L600p9sxv,像我这样的人 (Live),NaN,毛不易,1507534,001BHDR33FZVZ0,2196371,0001n7a82gh6IY,171,1501862400,像我这样的人,像我这样的人,明日之子 第8期,2017-08-05,2017,明日之子 第8期,PART 1,0.0
4,254554296,0,002XkEH930NXSr,一荤一素 (Live),NaN,毛不易,1507534,001BHDR33FZVZ0,10635028,002VxplL2gXAuH,314,1581609600,一荤一素,一荤一素,歌手·当打之年 第2期,2020-02-14,2020,歌手·当打之年 第2期,PART 1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,365425302,0,001LTTIo47jR0A,平凡的雕琢,《张卫国的夏天》电视剧主题曲,毛不易,1507534,001BHDR33FZVZ0,29168510,001NgEOG082AvL,250,1657987200,平凡的雕琢,平凡的雕琢,张卫国的夏天 电视剧原声带,2022-07-17,2022,张卫国的夏天 电视剧原声带,PART 10,9.0
96,218739560,0,002gveGh3VyYt5,项羽虞姬,《王者荣耀》项羽虞姬英雄主打歌,毛不易,1507534,001BHDR33FZVZ0,4867924,00436LYE43MyXL,291,1540483200,项羽虞姬,项羽虞姬,天美十年典藏：全明星音乐特辑,2018-10-26,2018,天美十年典藏：全明星音乐特辑,PART 10,9.0
97,413527489,0,001kC7KC04i6dR,小小的我,飞鹤528宝宝日主题曲,毛不易,1507534,001BHDR33FZVZ0,38472625,003pGxSJ3uVRka,210,1685235600,小小的我,小小的我,小小的我,2023-05-28,2023,小小的我,PART 10,9.0
98,424423033,0,000vA9oK3uS7CZ,看我始终,《镇魂街》第三季动画主题曲,毛不易,1507534,001BHDR33FZVZ0,42956068,002yWkME4OIJNh,189,1690560000,看我始终,看我始终,镇魂街第三季 动画原声带,2023-07-29,2023,镇魂街第三季 动画原声带,PART 10,9.0


In [77]:
df_songs_final.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)

# 测试

In [ ]:
1260/500

In [ ]:
2318/2.52

In [ ]:
from datetime import datetime
datetime.now().month